[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/03_analysis/C0_methods_overview.ipynb)

# C0: Core Methods Overview

---

## Learning Objectives

By the end of this notebook, you will understand:
1. **Table joins** - connecting permits, projects, parcels, and zoning
2. **Grouping and aggregation** - summarizing data by category
3. **Standard plots** - visualizations used throughout the project
4. **Reusable patterns** - code you can adapt for other analyses

This notebook is a **reference guide** - come back here when you need to remember how something works.

---

## 1. Setup

In [ ]:
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline

# Find project root
def find_project_root():
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if (path / '00_config').exists() and (path / 'modules').exists():
            return path
    return current

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

# Load config
config_path = ROOT / '00_config/berkeley_config.json'
if config_path.exists():
    with open(config_path) as f:
        CONFIG = json.load(f)
    DATA_DIR = ROOT / CONFIG['paths']['data_dir']
else:
    DATA_DIR = ROOT / 'data/processed'

print(f"Project root: {ROOT}")

In [ ]:
# Load the main housing projects dataset
df = pd.read_csv(DATA_DIR / 'housing_projects_FINAL.csv')
print(f"Loaded {len(df)} projects")

## 2. Joining Tables

In civic data, information is often split across multiple tables:
- **Projects** table: basic project info
- **Permits** table: permit numbers, dates, status
- **Parcels** table: APN, lot size, zoning
- **Events** table: timeline of actions

We use **joins** to connect these tables.

### Types of Joins

```
INNER JOIN: Only rows that match in BOTH tables
LEFT JOIN:  All rows from LEFT table, matching from RIGHT (or NULL)
RIGHT JOIN: All rows from RIGHT table, matching from LEFT (or NULL)
OUTER JOIN: All rows from BOTH tables
```

### Visual:
```
Projects Table           Parcels Table
┌────────┬────────┐      ┌────────┬──────────┐
│ id     │ apn    │      │ apn    │ lot_size │
├────────┼────────┤      ├────────┼──────────┤
│ 1      │ 123-45 │──────│ 123-45 │ 5000     │
│ 2      │ 234-56 │──────│ 234-56 │ 7500     │
│ 3      │ 345-67 │  ╳   │        │          │
└────────┴────────┘      └────────┴──────────┘

JOIN ON apn = apn
```

In [ ]:
# Example: Create two sample tables and join them

# Projects table (what we're building)
projects = pd.DataFrame({
    'project_id': [1, 2, 3],
    'address': ['123 Main St', '456 Oak Ave', '789 Pine Rd'],
    'apn': ['056-1234-001', '056-2345-002', '056-3456-003'],
    'units': [10, 25, 50]
})

# Parcels table (property info from assessor)
parcels = pd.DataFrame({
    'apn': ['056-1234-001', '056-2345-002', '056-9999-999'],  # Note: 056-3456-003 missing
    'lot_sqft': [5000, 7500, 3000],
    'zoning': ['R-2', 'C-2', 'R-1']
})

print("Projects table:")
display(projects)
print("\nParcels table:")
display(parcels)

In [ ]:
# INNER JOIN: Only matching rows
# Use case: "Show me projects where I have parcel data"

inner_result = pd.merge(projects, parcels, on='apn', how='inner')

print("INNER JOIN result (only matches):")
print(f"Projects: 3, Parcels: 3, Result: {len(inner_result)}")
display(inner_result)

In [ ]:
# LEFT JOIN: All projects, even if no parcel match
# Use case: "Show me ALL projects, with parcel info where available"

left_result = pd.merge(projects, parcels, on='apn', how='left')

print("LEFT JOIN result (all projects):")
print(f"Projects: 3, Parcels: 3, Result: {len(left_result)}")
display(left_result)

# Note: Project 3 has NaN for parcel data (no match found)

In [ ]:
# Reusable pattern for joining

def join_tables(left_df, right_df, join_key, how='left'):
    """
    Join two DataFrames with logging.
    
    Parameters:
        left_df: Primary table (keep all rows if how='left')
        right_df: Secondary table (lookup data)
        join_key: Column name to join on (must exist in both)
        how: 'inner', 'left', 'right', or 'outer'
    
    Returns:
        Joined DataFrame
    """
    result = pd.merge(left_df, right_df, on=join_key, how=how)
    
    print(f"Join summary:")
    print(f"  Left table:  {len(left_df)} rows")
    print(f"  Right table: {len(right_df)} rows")
    print(f"  Result:      {len(result)} rows")
    
    # Check for unmatched
    if how == 'left':
        unmatched = result[right_df.columns[1]].isna().sum()
        print(f"  Unmatched:   {unmatched} rows")
    
    return result

# Example usage:
joined = join_tables(projects, parcels, 'apn', how='left')

## 3. Grouping and Aggregation

Grouping lets us answer questions like:
- "How many units per year?"
- "Average project size by zoning district?"
- "Total projects by status?"

### The Pattern
```python
df.groupby('category_column')['value_column'].aggregation_function()
```

### Common Aggregations
- `sum()` - total
- `count()` - number of rows
- `mean()` - average
- `median()` - middle value
- `min()`, `max()` - extremes

In [ ]:
# Units by year
if 'year' in df.columns and 'net_units' in df.columns:
    units_by_year = df.groupby('year')['net_units'].sum()
    print("Total units by year:")
    print(units_by_year)
else:
    print("Required columns not found")

In [ ]:
# Projects by status
if 'status' in df.columns:
    projects_by_status = df.groupby('status').size()
    print("Projects by status:")
    print(projects_by_status.sort_values(ascending=False))

In [ ]:
# Multiple aggregations at once
if 'status' in df.columns and 'net_units' in df.columns:
    summary = df.groupby('status').agg({
        'net_units': ['count', 'sum', 'mean']
    }).round(1)
    
    # Flatten column names
    summary.columns = ['num_projects', 'total_units', 'avg_units']
    
    print("Status summary:")
    display(summary.sort_values('total_units', ascending=False))

## 4. Standard Plots

Here are reusable plotting functions used throughout the project.

In [ ]:
def plot_units_by_year(df, year_col='year', units_col='net_units', title='Housing Units by Year'):
    """
    Create a bar chart of housing units by year.
    
    Parameters:
        df: DataFrame with year and units columns
        year_col: Name of year column
        units_col: Name of units column
        title: Chart title
    """
    # Group by year
    by_year = df.groupby(year_col)[units_col].sum()
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Bar chart
    bars = by_year.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    
    # Labels
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('Housing Units', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    # Add value labels
    for i, v in enumerate(by_year):
        ax.text(i, v + max(by_year)*0.02, f'{int(v):,}', ha='center', fontsize=9)
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    return fig, ax


# Use the function
if 'year' in df.columns and 'net_units' in df.columns:
    fig, ax = plot_units_by_year(df)
    plt.show()

In [ ]:
def plot_units_by_status(df, status_col='status', units_col='net_units', title='Housing Units by Status'):
    """
    Create a horizontal bar chart of units by status.
    
    Parameters:
        df: DataFrame with status and units columns
        status_col: Name of status column
        units_col: Name of units column
        title: Chart title
    """
    # Group by status
    by_status = df.groupby(status_col)[units_col].sum().sort_values()
    
    # Create figure
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Horizontal bar chart
    colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(by_status)))
    by_status.plot(kind='barh', ax=ax, color=colors)
    
    # Labels
    ax.set_xlabel('Housing Units', fontsize=12)
    ax.set_ylabel('Status', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    # Add value labels
    for i, v in enumerate(by_status):
        ax.text(v + max(by_status)*0.02, i, f'{int(v):,}', va='center', fontsize=9)
    
    plt.tight_layout()
    
    return fig, ax


# Use the function
if 'status' in df.columns and 'net_units' in df.columns:
    fig, ax = plot_units_by_status(df)
    plt.show()

## 5. Using Project Modules

The `modules/` directory contains reusable Python code. Here's how to use them:

In [ ]:
# Import modules (if they exist)
try:
    from modules.data_loader import load_csv
    print("data_loader module available")
except ImportError:
    print("data_loader module not found")

try:
    from modules.address_normalizer import normalize_address
    print("address_normalizer module available")
    
    # Demo
    test_address = "123 MAIN STREET"
    normalized = normalize_address(test_address)
    print(f"  Example: '{test_address}' -> '{normalized}'")
except ImportError:
    print("address_normalizer module not found")

try:
    from modules.geocoder import geocode_from_lookup
    print("geocoder module available")
except ImportError:
    print("geocoder module not found")

---

## Summary: Core Patterns

### Joining Tables
```python
result = pd.merge(left_df, right_df, on='key_column', how='left')
```

### Grouping and Aggregation
```python
df.groupby('category')['value'].sum()     # Total by category
df.groupby('category').size()              # Count by category
df.groupby('category')['value'].mean()     # Average by category
```

### Plotting
```python
series.plot(kind='bar')   # Vertical bars
series.plot(kind='barh')  # Horizontal bars
series.plot(kind='line')  # Line chart
series.plot(kind='pie')   # Pie chart
```

### Using Modules
```python
from modules.module_name import function_name
result = function_name(arguments)
```

---

Bookmark this notebook and return when you need to remember a pattern!